# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

F. Zagaria  ->  F. Zagaria  |  ['F. Zagaria']
K. Doi  ->  K. Doi  |  ['K. Doi']
M. Benisty  ->  M. Benisty  |  ['M. Benisty']
P. Molliere  ->  P. Molliere  |  ['P. Molliere']
S. Kraus  ->  S. Kraus  |  ['S. Kraus']
M. Zhang  ->  M. Zhang  |  ['M. Zhang']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
J. Li  ->  J. Li  |  ['J. Li']
Y. Wang  ->  Y. Wang  |  ['Y. Wang']
P. J. Smith  ->  P. J. Smith  |  ['P. J. Smith']
N. Neumayer  ->  N. Neumayer  |  ['N. Neumayer']
C. Clontz  ->  C. Clontz  |  ['C. Clontz']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']


T. Harvey  ->  T. Harvey  |  ['T. Harvey']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
T. Harvey  ->  T. Harvey  |  ['T. Harvey']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Paterson  ->  K. Paterson  |  ['K. Paterson']
M. Schirmer  ->  M. Schirmer  |  ['M. Schirmer']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
Y.-S. Ting  ->  Y.-S. Ting  |  ['Y.-S. Ting']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
J. Li  ->  J. Li  |  ['J. Li']
T. Henning  ->  T. Henning  |  ['T. Henning']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Paterson  ->  K. Paterson  |  ['K. Paterson']
M. Schirmer  ->  M. Schirmer  |  ['M. Schirmer']


Y. Wang  ->  Y. Wang  |  ['Y. Wang']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Paterson  ->  K. Paterson  |  ['K. Paterson']
M. Schirmer  ->  M. Schirmer  |  ['M. Schirmer']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Paterson  ->  K. Paterson  |  ['K. Paterson']
M. Schirmer  ->  M. Schirmer  |  ['M. Schirmer']
J. Matharu  ->  J. Matharu  |  ['J. Matharu']
T. Harvey  ->  T. Harvey  |  ['T. Harvey']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
M. Schirmer  ->  M. Schirmer  |  ['M. Schirmer']
K. Paterson  ->  K. Paterson  |  ['K. Paterson']
J. E. Davies  ->  J. E. Davies  |  ['J. E. Davies']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
M. Schirmer  ->  M. Schirmer  |  ['M. Schirmer']
K. Paterson  ->  K. Paterson  |  ['K. Paterson']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Paterson  ->  K. Paterson  |  ['K. Pater

K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
K. Paterson  ->  K. Paterson  |  ['K. Paterson']
M. Schirmer  ->  M. Schirmer  |  ['M. Schirmer']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
Arxiv has 218 new papers today
          34 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/34 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.14632


extracting tarball to tmp_2611.14632... done.
Retrieving document from  https://arxiv.org/e-print/2611.14646


extracting tarball to tmp_2611.14646... done.


P. Molliere  ->  P. Molliere  |  ['P. Molliere']
Retrieving document from  https://arxiv.org/e-print/2611.14723


extracting tarball to tmp_2611.14723...

... (96 outputs left out)

/tmp/ipykernel_2754/2822249172.py:52: LatexWarning: 2611.17040 did not run properly
list index out of range
  warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +


Retrieving document from  https://arxiv.org/e-print/2611.17305
extracting tarball to tmp_2611.17305...

 done.


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.17305/main.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'authors' from 'tmp_2611.17305/authors.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.14646-b31b1b.svg)](https://arxiv.org/abs/2611.14646) | **Signatures of CH$_4$ and NH$_3$ in the atmosphere of $β$ Pictoris b**  |
|| A. Barone, et al. -- incl., <mark>P. Molliere</mark> |
|*Appeared on*| *2026-11-13*|
|*Comments*| *Accepted for publication in Astronomy & Astrophysics. 7 pages, 3 figures*|
|**Abstract**|            $\beta$ Pictoris is a nearby (19.7 pc), young ($\sim$ 24 Myr), bright ($V = 3.85$) planetary system hosting three directly imaged planets. Previous high-resolution spectroscopy of $\beta$ Pic b has detected H$_2$O and CO, but not CH$_4$ or NH$_3$. Combining observations across a broad wavelength range has been shown to maximise sensitivity to such weak molecular signals. We searched for previously undetected molecular species in the atmosphere of $\beta$ Pic b using high-resolution near-infrared spectra. We combined 44 CRIRES+ spectral settings covering the H, K, L, and M bands and cross-correlated the extracted planetary spectra with molecular templates. For species with uncertain detectability, we weighted the observations by their molecular information content and data quality, independent of the measured candidate-species cross-correlation functions. We recover H$_2$O and CO at cross-correlation S/N values of approximately 200 and 50, respectively. We additionally identify tentative CH$_4$ and NH$_3$ signals near the expected radial velocity of $\beta$ Pic b, with S/N values of approximately 5.3 and 5.7. Equivalent extractions at the same projected separation on the opposite side of the star show no comparable peaks. We classify the CH$_4$ and NH$_3$ signals as tentative because their cross-correlation S/N values do not fully account for possible residual systematics. If confirmed, these would be the first detections of both species in the atmosphere of $\beta$ Pic b.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.15883-b31b1b.svg)](https://arxiv.org/abs/2611.15883) | **Euclid Data Release 1 (DR1). Euclid Lens Identification Pipeline and Scientific Exploration (ECLIPSE) A: Strong lens catalogue overview**  |
|| E. Collaboration, et al. -- incl., <mark>T. Harvey</mark>, <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-13*|
|*Comments*| *28 pages. Submitted to A&A as part of the Special Issue "Euclid First Data Release"*|
|**Abstract**|            We present 15 072 high-confidence strong lens candidates in Euclid Data Release 1 (DR1), spanning 2000 deg$^2$. These comprise 6073 grade A ('sure lens') and 8999 grade B ('probable lens') systems, which we publish alongside a further 9010 grade C ('possible lens') candidates. With more high-confidence candidates than all previous catalogues combined, this sample establishes a large, homogeneous data set of galaxy-scale strong lenses with high-resolution imaging, transforming strong lensing from a field of rare discoveries to a robust statistical tool for astrophysics and cosmology. Of the roughly 900 million sources in DR1, we searched the 68 million passing our catalogue selection using a bespoke discovery pipeline that synthesises scores from five machine-learning models with visual classifications from around 7000 citizen scientists and 67 strong lensing experts. Beyond classical Einstein rings, our catalogue includes 84 candidate double-source-plane lenses and over 1000 candidate disc lenses, two exotic classes with few previously known examples, as well as several hundred candidate lensed AGN and quasars. We deliver automated lens models for the combined grade A+B systems, and photometric redshifts and stellar population properties for the deflector galaxies, providing immediate science-ready data products. We find that the majority of the deflector galaxies are massive and quiescent, with stellar masses of $\log_{10}(M_\ast /M_\odot) = 11.39^{+0.29}_{-0.33}$ (Salpeter initial mass function; 16th-84th percentile range). The deflector galaxies span a range of photometric redshifts ($z=0.68 ^{+0.33}_{-0.29}$), extending to higher redshifts than previous ground-based lens samples, and the lenses have a typical Einstein radius of $0.85 ^{+0.44}_{-0.24}$ arcsec. Extrapolating by area, we expect over 100 000 high-confidence strong lens candidates in the Euclid Wide Survey.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.15891-b31b1b.svg)](https://arxiv.org/abs/2611.15891) | **Euclid Data Release 1 (DR1). NISP spectroscopy of gravitational arcs and multiple images**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-13*|
|*Comments*| *Paper intended for the A&A Special Issue "Euclid First Data Release". 15 pages, 5 figures*|
|**Abstract**|            Applications of strong gravitational lensing in astrophysics and cosmology critically depend on building reliable strong lensing models, which require accurate measurements of on-sky positions of multiple images and of the redshifts of the background sources from which they originate. In this context, Euclid is a unique facility, as it simultaneously provides high-resolution multi-band photometric imaging and slitless spectroscopy for an unprecedentedly large sample of gravitational lenses, most of which are newly discovered. In this work, we exploit Euclid slitless spectroscopic data to determine the redshifts of four spatially extended lensed sources, and to reconstruct emission-line images. Specifically, two of the target sources are lensed into giant arcs by the massive galaxy clusters SPT-CLJ0411$-$4819 ($z_\mathrm{d}=0.424$, where "d" denotes the lensing deflector) and Abell 2280 ($z_\mathrm{d}=0.326$). The third system is a quadruply imaged active galactic nucleus (AGN) candidate strongly lensed by a massive galaxy at $z_\mathrm{d}=0.68$, while the fourth is a galaxy-scale Einstein ring produced by a lens galaxy at $z_\mathrm{d}=0.235$. This work represents a proof of concept for the exploitation of Euclid slitless spectroscopic data in the study of strong gravitational lensing, providing one of the first determinations of spectroscopic redshifts for strongly lensed features from NISP spectra. These results demonstrate the feasibility of applying the same methodology to the very large sample of gravitational lenses that Euclid has started to discover and is expected to identify over the course of the mission.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.16077-b31b1b.svg)](https://arxiv.org/abs/2611.16077) | **Desorption of water from cosmic silicates**  |
|| C. Jäger, D. Köhler, <mark>T. Henning</mark>, F. Langenhorst |
|*Appeared on*| *2026-11-13*|
|*Comments*| **|
|**Abstract**|            Recent JWST and ALMA observations of protoplanetary disks have identified water reservoirs in the inner regions of the disks. This would provide terrestrial planets forming therein access to a water reservoir. In situ formation of water molecules on grain surfaces was proposed as one possible formation process in the PDS 70 system, but transport by small silicate particles coupled to the gas could be another mechanism. Laboratory experiments on the thermally induced desorption of water molecules from different water-containing silicates at elevated temperatures provide another possibility for the enrichment of water molecules in the inner disk. Physically and chemically bound water in silicates can be released upon annealing processes. Water that is physically bound undergoes desorption at temperatures up to 473 K. In contrast, chemically bound water in form of different silanol or Mg-OH groups in amorphous silicates or phyllosilicates, can be released at higher temperatures, ranging from 773 to 923 K. The relevant temperatures are sufficient to trigger the solid-state recrystallization of amorphous silicates to Mg-rich orthopyroxene and olivine. The existence of these crystalline silicates has been substantiated through Spitzer and JWST-MIRI observations. Based on the experimental results, we postulate that the desorption of bound hydroxyl groups may serve as a mechanism for the formation of the water reservoir at temperatures up to 900 K, much higher than the water sublimation temperature of 100 to 200 K.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.16444-b31b1b.svg)](https://arxiv.org/abs/2611.16444) | **Euclid Data Release 1 (DR1). The PHZ processing function for the physical properties of galaxies**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-13*|
|*Comments*| *13 pages, 8 figures. Paper intended for the A\&A Special Issue "Euclid First Data Release''*|
|**Abstract**|            Euclid is a European Space Agency mission mapping the extragalactic sky, aiming to provide the most-accurate 3D view of the Universe at optical and near-infrared wavelengths, by measuring the photometric redshifts of billions of galaxies. Within the Euclid Consortium, this task is executed by the PHZ Organisation Unit. PHZ takes as input the multi-wavelength photometry of Euclid and ground-based data, and returns as output the source classification, photometric redshifts for the Euclid core science (i.e., cosmology), and photometric redshifts plus physical parameters and absolute magnitudes for non-cosmological science. As part of the first Euclid Data Release (DR1), we describe the details of the DR1 pipeline for non-cosmological science, along with the released data products, and their validation. The non-cosmological science data products are obtained with NNPZ, a nearest-neighbour algorithm developed within PHZ that infers the properties of each target galaxy from its closest matches, in colour-magnitude space, within a reference sample of galaxies with known redshifts and physical parameters. We built this reference sample from Euclid observations of COSMOS, one of the Euclid Auxiliary Fields, labelling each galaxy with redshifts and physical parameters derived from the COSMOS2020 multi-wavelength photometry, consistently with the Euclid core-science results. Combined with the adjustments and priors required to generalise the results across the entire extragalactic sky, this approach yields high-quality and internally consistent data products. We tested the performance using both internal metrics and comparisons to ancillary data sets (e.g., COSMOS2020, 3D-HST, and DESI) and scaling relations (e.g., the star-forming main sequence); the results are in line with expectations and pre-DR1 simulations for both the Euclid Wide Survey and Euclid Deep Fields.         |

... (26 outputs left out)


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.16548-b31b1b.svg)](https://arxiv.org/abs/2611.16548) | **Euclid Data Release 1 (DR1): Spacecraft performance in the L2 environment**  |
|| E. Collaboration, et al. -- incl., <mark>M. Schirmer</mark>, <mark>K. Paterson</mark>, <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-13*|
|*Comments*| *38 pages, 33 figures; companion paper of Euclid DR1*|
|**Abstract**|            The Euclid mission was designed around the need for thermo-optical and mechanical stability, stray-light suppression, low outgassing, and precise guiding. After three years in space, a wealth of data is available to assess the performance in these areas. We review the L2 halo-orbit design and consumables budget. Solar stray-light gradually increases by 14\% per year and is consistent with meteoroid puncturing, but remains low overall. Data loss from high solar activity has declined notably since the maximum of the 25th Solar cycle passed. The thermodynamical response of the optical elements to a thermal step impulse follows sigmoidal growth curves, entering the asymptotic equilibration regime after 21--46 hours. Water-ice contamination has affected data from both instruments and requires continued monitoring; we review detection and decontamination methods, and Euclid's contamination history. High-precision measurements of the zodiacal background reveal, at eight sigma significance, the subtle modulation caused by the changing perspective on the dust distribution due to the L2 halo-orbit libration. These data also show that conservative pre-launch estimates overestimated the out-of-field stray-light by factors of 4 (H band) to 27 (I band). Telemetry data demonstrate excellent guiding, and enable highly significant detections of solar radiation pressure and of thermal-emission recoil from Euclid's unilluminated dark sides. Lastly, we present a list of 34 meteoroid impacts, discuss their mechanical and optical signatures, and derive a minimum detectable impactor mass of 5e-7 g.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.16602-b31b1b.svg)](https://arxiv.org/abs/2611.16602) | **Euclid Data Release 1 (DR1) Euclid photometric masks**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-13*|
|*Comments*| **|
|**Abstract**|            This paper describes the Euclid DR1 photometric masks, which trace spatial variations in the survey properties (e.g. satellite position angles, integration time, sky tiling, and survey footprint); in the instrumental performance of Euclid and the external data (EXT) it uses (e.g. survey coverage, depth, PSF width, quality flags, systematic effects and astrometric uncertainties); and in possible contamination from sky emission (e.g. galactic extinction and zodiacal light emission). Among this, the effective coverage mask is particularly important as it defines the effective survey area and can be used to exclude spurious objects when selecting Euclid photometric samples for scientific analyses. The Euclid photometric masks are built for each observed sky region (tile) and are provided as full-sky HEALPix maps (nested ordering and explicit indexing) at Nside = 16 384, equating to a resolution of 12"88, for both the Euclid Wide Survey (EWS) and Euclid Deep Survey (EDS). We estimate the total area of the DR1 EWS (EDS) to be 2008.8 (74.7) deg$^2$ and find that 82 (81)\% of this area is preserved by the effective coverage mask. We observe that Euclid's instrumental properties are consistent with expectations, showing only minor relative variations across the sky. For DR1 EWS the standard deviation of the depth (computed for a 2" aperture and at 5 $\sigma$) is below 0.2 AB mag, with median values of 25.69, 24.14, 24.3, and 24.29 for the Euclid IE, YE, JE, and HE bands. Finally, we expect these photometric masks to be key to many science applications and in particular for cosmological analyses.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.16766-b31b1b.svg)](https://arxiv.org/abs/2611.16766) | **Euclid Data Release 1 (DR1). Persistence in NISP detectors**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark>, <mark>K. Paterson</mark>, <mark>M. Schirmer</mark> |
|*Appeared on*| *2026-11-13*|
|*Comments*| **|
|**Abstract**|            Detector persistence is a significant source of systematic error in infrared observations and affects the scientific performance of the \Euclid Near Infrared Spectrometer and Photometer (NISP). We characterise persistence using complementary ground calibration and in-flight data, develop empirical models of its temporal evolution, and assess its impact on science observations and the \Euclid data-processing pipeline. Persistence is highly non-uniform across the focal plane, with variations of up to two orders of magnitude between and within detectors. Its amplitude is typically below 1\% of the stimulus signal and depends nonlinearly on the preceding fluence. The decay rate is faster for brighter sources. During mission operations, the persistence signal has progressively decreased by roughly 10\%, while the decay rate has remained approximately stable. Persistence following over-saturating signals does not follow the empirical relations derived for under-saturating fluences and plausibly reduces persistence response over longer timescales. The impact of charge trapping and release on flat-field observations and on the background in science exposures is also discussed. Persistence measured in science observations is observed to differ systematically from that derived from dark calibration data. In addition, persistence of energetic particle hits is observed to be different from that of astronomical sources of comparable signal. Finally, we discuss the impact of persistence on \Euclid NISP observations and describe the masking strategy adopted for the Q1 abd DR1 releases, focusing on identifying which pixels are affected by persistence and for how long they should be excluded from science observations.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.17040-b31b1b.svg)](https://arxiv.org/abs/2611.17040) | **Euclid Data Release 1 (DR1): The Euclid surveys at DR1**  |
|| E. Collaboration, et al. -- incl., <mark>K. Paterson</mark>, <mark>M. Schirmer</mark> |
|*Appeared on*| *2026-11-13*|
|*Comments*| *Paper intended for the A&A Special Issue 'Euclid First data release'*|
|**Abstract**|            Euclid is a surveying mission that is progressively imaging and spectroscopically mapping a wide area of extragalactic sky in visible and near infrared bands. The Euclid mission is, however, not restricted to the Euclid Wide Survey, but it includes data from various other survey components. This paper presents the Data Release 1 (DR1) status of the multiple Euclid surveys and fields, including the Euclid Deep Survey, Euclid Auxiliary Fields, calibration fields, the Euclid Medium-Deep Survey, the Euclid Ecliptic Survey, and the Independent Legacy Scientists' field, in addition to the Euclid Wide Survey. Some of these fields were observed during the initial Performance-Verification phase, others during the Phase-Diversity calibration campaign, and the majority during the subsequent science operations phase. This paper describes the different procedures of survey implementation applied in the various phases of the mission, from the set of requirements, constraints, and limitations, to the final operational products used to operate the spacecraft and execute the science surveys. The resulting main component of the Euclid surveys, the Euclid Wide Survey, covers at DR1 an area of 2100$\,{\rm deg}^2$, in two separate regions, of which 1900$\,{\rm deg}^2$ corresponds to data validated in higher-level processing.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.14646/./Figures/CCF_H2O_7.00e-04.png', 'tmp_2611.14646/./Figures/CCF_CO_3.00e-06.png', 'tmp_2611.14646/./Figures/CCF_CH4_5.00e-05.png', 'tmp_2611.14646/./Figures/CCF_NH3_5.00e-05.png', 'tmp_2611.14646/./Figures/Bands.png', 'tmp_2611.14646/./Figures/Threshold.png']
copying  tmp_2611.14646/./Figures/CCF_H2O_7.00e-04.png to _build/html/
copying  tmp_2611.14646/./Figures/CCF_CO_3.00e-06.png to _build/html/
copying  tmp_2611.14646/./Figures/CCF_CH4_5.00e-05.png to _build/html/
copying  tmp_2611.14646/./Figures/CCF_NH3_5.00e-05.png to _build/html/
copying  tmp_2611.14646/./Figures/Bands.png to _build/html/
copying  tmp_2611.14646/./Figures/Threshold.png to _build/html/
exported in  _build/html/2611.14646.md
    + _build/html/tmp_2611.14646/./Figures/CCF_H2O_7.00e-04.png
    + _build/html/tmp_2611.14646/./Figures/CCF_CO_3.00e-06.png
    + _build/html/tmp_2611.14646/./Figures/CCF_CH4_5.00e-05.png
    + _build/html/tmp_2611.14646/./Figures/CCF_NH3_5.00e-05.png
    + _bu

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\Pic}{\beta~Pic\xspace}$
$\newcommand{\water}{H_2O\xspace}$
$\newcommand{\ammonia}{NH_3\xspace}$
$\newcommand{\methane}{CH_{4}\xspace}$</div>



<div id="title">

# Signatures of $\methane$ and $\ammonia$ in the atmosphere of $\beta$ Pictoris b

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.14646-b31b1b.svg)](https://arxiv.org/abs/2611.14646)<mark>Appeared on: 2026-11-13</mark> -  _Accepted for publication in Astronomy & Astrophysics. 7 pages, 3 figures_

</div>
<div id="authors">

A. Barone, et al. -- incl., <mark>P. Molliere</mark>

</div>
<div id="abstract">

**Abstract:** $\beta$ Pictoris is a nearby (19.7 pc), young ( $\sim$ 24 Myr), bright ( $V = 3.85$ ) planetary system hosting three directly imaged planets. Previous high-resolution spectroscopy of $\Pic$ b has detected $\water$ and CO, but not $\methane$ or $\ammonia$ . Combining observations across a broad wavelength range has been shown to maximise sensitivity to such weak molecular signals. We searched for previously undetected molecular species in the atmosphere of $\Pic$ b using high-resolution near-infrared spectra. We combined 44 CRIRES+ spectral settings covering the H, K, L, and M bands and cross-correlated the extracted planetary spectra with molecular templates. For species with uncertain detectability, we weighted the observations by their molecular information content and data quality, independent of the measured candidate-species cross-correlation functions. We recover $\water$ and CO at cross-correlation S/N values of approximately 200 and 50, respectively. We additionally identify tentative $\methane$ and $\ammonia$ signals near the expected radial velocity of $\Pic$ b, with S/N values of approximately 5.3 and 5.7. Equivalent extractions at the same projected separation on the opposite side of the star show no comparable peaks. We classify the $\methane$ and $\ammonia$ signals as tentative because their cross-correlation S/N values do not fully account for possible residual systematics. If confirmed, these would be the first detections of both species in the atmosphere of $\Pic$ b.

</div>

<div id="div_fig1">

<img src="tmp_2611.14646/./Figures/CCF_H2O_7.00e-04.png" alt="Fig3.1" width="25%"/><img src="tmp_2611.14646/./Figures/CCF_CO_3.00e-06.png" alt="Fig3.2" width="25%"/><img src="tmp_2611.14646/./Figures/CCF_CH4_5.00e-05.png" alt="Fig3.3" width="25%"/><img src="tmp_2611.14646/./Figures/CCF_NH3_5.00e-05.png" alt="Fig3.4" width="25%"/>

**Figure 3. -** Cross-correlation functions of $\Pic$ b with an $\water$(upper-left panel), CO (upper-right panel), $\methane$(bottom-left panel), and $\ammonia$(bottom-right panel). The black lines are the cross-correlation functions at $\Pic$ b's location; the grey lines are the corresponding functions at the same projected separation on the opposite side of the star; meanwhile, the brown lines show the residuals after subtraction of the scaled model autocorrelation from the planetary CCF.
     (*fig: CCFs*)

</div>
<div id="div_fig2">

<img src="tmp_2611.14646/./Figures/Bands.png" alt="Fig1" width="100%"/>

**Figure 1. -** From top to bottom: spectral ranges covered by detector orders in the grating settings used for this analysis, divided into H, K, L and M bands, respectively. The observations cover almost all of the four bands. The overplotted black line is the fractional atmospheric transmission. These plots were generated using \texttt{CRIRESViz}$\footnote$mark (*fig: Orders*)

</div>
<div id="div_fig3">

<img src="tmp_2611.14646/./Figures/Threshold.png" alt="Fig2" width="100%"/>

**Figure 2. -** The upper panel shows, in blue, the simulated planetary flux containing $\methane$ and produced using \texttt{petitRadTRANS}. The model is baseline-subtracted. The bottom panel shows the planetary contrast, in yellow, derived from the upper panel and used in the cross-correlation procedure. The black dashed line represents the threshold (i.e. 0.1 in contrast space) used to select the regions containing information. The green areas denote where the model's gradient exceeds the threshold (selected regions), while the red ones are discarded regions. (*fig: threshold*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.14646"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\tinier}{\fontsize{7.5pt}{7.5pt}\selectfont}$
$\newcommand{\purple}[1]{\textcolor{purple}{#1}}$
$\newcommand{\orange}[1]{\textcolor{orange}{#1}}$
$\newcommand{\red}[1]{\textcolor{black}{#1}}$
$\newcommand{\tom}[1]{\textcolor{purple}{#1}}$
$\newcommand{\phil}[1]{\textcolor{orange}{#1}}$
$\newcommand{\phildone}[1]{\textcolor{black}{#1}}$
$\newcommand{\black}[1]{\textcolor{black}{#1}}$
$\newcommand{\blue}[1]{\textcolor{blue}{#1}}$
$\newcommand{\comments}[1]{\textcolor{black}{#1}}$
$\newcommand{\fixed}[1]{\textcolor{black}{#1}}$
$\newcommand{\old}{^{\star}}$
$\newcommand{\gradeA}{\num{6073}\xspace}$
$\newcommand{\gradeB}{\num{8999}\xspace}$
$\newcommand{\gradeC}{\num{9010}\xspace}$
$\newcommand{\gradeX}{\num{2580}\xspace}$
$\newcommand{\gradeAB}{\num{15072}\xspace}$
$\newcommand{\gradeABC}{\num{24082}\xspace}$
$\newcommand{\gradeABCX}{\num{26662}\xspace}$
$\newcommand{\dspl}{84\xspace}$
$\newcommand{\discs}{1000\xspace}$
$\newcommand{\lqso}{300\xspace}$
$\newcommand{\angelosmodels}{\num{9538}\xspace}$
$\newcommand{\randoms}{\num{50000}\xspace}$
$\newcommand{\drtwo}{\num{60000}\xspace}$
$\newcommand{\ews}{\num{110000}\xspace}$
$\newcommand{\dronearea}{\num{2000}\xspace}$
$\newcommand{\experts}{\num{67}\xspace}$
$\newcommand{\knownA}{\num{417}\xspace}$
$\newcommand{\knownB}{\num{161}\xspace}$
$\newcommand{\knownC}{\num{55}\xspace}$
$\newcommand{\stellarmass}{\logten (M_\ast /M_\odot) = 11.39^{+0.29}_{-0.33}\xspace}$
$\newcommand{\ssfr}{\log_{10}(\textrm{sSFR/yr}^{-1}) = -11.25^{+0.57}_{-0.37}\xspace}$
$\newcommand{\photoz}{0.68 ^{+0.33}_{-0.29}\xspace}$
$\newcommand{\thetaE}{0.85 ^{+0.44}_{-0.24}\xspace}$
$\newcommand{\orcid}[1]{ \orcidlink{#1}}$
$\newcommand{\linenumbers}[0]\usepackage$
$\newcommand{\thefigure}{A.\arabic{figure}}$
$\newcommand{\thefigure}{A.\arabic{figure}}$
$\newcommand{\thefigure}{B.\arabic{figure}}$
$\newcommand{\thefigure}{C.\arabic{figure}}$
$\newcommand{\thefigure}{C.\arabic{figure}}$</div>



<div id="title">

# Euclid Data Release 1 (DR1): Euclid Lens Identification Pipeline and Scientific Exploration (ECLIPSE) A: Strong lens catalogue overview$\thanks{Results in this paper were obtained with the help of the volunteers of the Space Warps project whose contributions are individually acknowledged on the Space Warps\href{https://www.zooniverse.org/projects/aprajita/space-warps-esa-euclid/about/team}{`The Team'} webpage and in Appendix \ref{app:sw}.}$

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.15883-b31b1b.svg)](https://arxiv.org/abs/2611.15883)<mark>Appeared on: 2026-11-13</mark> -  _28 pages. Submitted to A&A as part of the Special Issue "Euclid First Data Release"_

</div>
<div id="authors">

E. Collaboration, et al. -- incl., <mark>T. Harvey</mark>, <mark>K. Jahnke</mark>

</div>
<div id="abstract">

**Abstract:** We present $\fixed{\gradeAB}$ high-confidence strong lens candidates in Euclid Data Release 1 (DR1), spanning $\fixed{\dronearea}$ deg $^2$ . These comprise $\fixed{\gradeA}$ grade A (`sure lens') and $\fixed{\gradeB}$ grade B (`probable lens') systems, which we publish alongside a further $\fixed{\gradeC}$ grade C (`possible lens') candidates.With more high-confidence candidates than all previous catalogues combined, this sample establishes a large, homogeneous data set of galaxy-scale strong lenses with high-resolution imaging, transforming strong lensing from a field of rare discoveries to a robust statistical tool for astrophysics and cosmology.Of the roughly 900 million sources in DR1, we searched the 68 million passing our catalogue selection using a bespoke discovery pipeline that synthesises scores from $\fixed{five}$ machine-learning models with visual classifications from around 7000 citizen scientists and $\fixed{\experts}$ strong lensing experts.Beyond classical Einstein rings, our catalogue includes $\fixed{\dspl}$ candidate double-source-plane lenses and over $\fixed{\discs}$ candidate disc lenses, two exotic classes with few previously known examples, as well as several hundred candidate lensed AGN and quasars.We deliver automated lens models for the combined grade A+B systems, and photometric redshifts and stellar population properties for the deflector galaxies, providing immediate science-ready data products.We find $\comments{that}$ the majority of the deflector galaxies are massive and quiescent, with $\comments{stellar masses of}$ $\red{\stellarmass}$ $\comments{(assuming a Salpeter initial mass function; the range corresponding to the$16$th--$84$th percentiles)}$ .The $\comments{deflector galaxies span a range of photometric redshifts ($z=$ \red{\photoz})}$ , extending to higher redshifts than previous ground-based lens samples, and $\comments{the lenses have}$ a $\comments{typical}$ Einstein radius of $\red{\thetaE}$ arcsec.Extrapolating by area, we expect over $\num{100000}$ high-confidence strong lens $\comments{candidates}$ in the Euclid Wide Survey.

</div>

<div id="div_fig1">

<img src="tmp_2611.15883/./fig/lenses_uniform/Tile102029816RA0507163409612DECNEG0397694014021_filled.jpg" alt="Fig6.1" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102009162RA0450319776694DECNEG0628849505935_filled.jpg" alt="Fig6.2" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/EUCL-DR1-L001604.jpg" alt="Fig6.3" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102012028RA0853629092626DECNEG0587974678810_mtf_vis_h.jpg" alt="Fig6.4" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102016447RA0620944429136DECNEG0534539081873_mtf_vis_y.jpg" alt="Fig6.5" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/structure.jpg" alt="Fig6.6" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102024485RA0561462671450DECNEG0447956407561_filled.jpg" alt="Fig6.7" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102024487RA0573720545754DECNEG0449051637754_filled.jpg" alt="Fig6.8" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102030933RA0506001865396DECNEG0390131023333_filled.jpg" alt="Fig6.9" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102032631RA0456822460276DECNEG0372568265017_filled.jpg" alt="Fig6.10" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102031542RA0784672633839DECNEG0382766418062_filled.jpg" alt="Fig6.11" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102014712RA0218237847000DECNEG0552859379000_filled.jpg" alt="Fig6.12" width="8%"/>

**Figure 6. -** Selection of more unusual lenses in DR1. (*fig:misc2*)

</div>
<div id="div_fig2">

<img src="tmp_2611.15883/./fig/lenses/RA084p7283586_DECm42p3133523_STCI_BGrestored_HYVIS_10x10arcsec_16bit.jpg" alt="Fig5.1" width="8%"/><img src="tmp_2611.15883/./fig/lenses/Tile102033812RA0612063355755DECNEG0364248198545_asinh_vis_y.jpg" alt="Fig5.2" width="8%"/><img src="tmp_2611.15883/./fig/lenses/Tile102010191RA0565900502093DECNEG0617079579921_filled.jpg" alt="Fig5.3" width="8%"/><img src="tmp_2611.15883/./fig/lenses/Tile102012773RA0694785521974DECNEG0578484437685_mtf_vis_y.jpg" alt="Fig5.4" width="8%"/><img src="tmp_2611.15883/./fig/lenses/qso1_mtf2.jpg" alt="Fig5.5" width="8%"/><img src="tmp_2611.15883/./fig/lenses/Tile102019108RA0450043260952DECNEG0503940818494_mtf_vis_y.jpg" alt="Fig5.6" width="8%"/><img src="tmp_2611.15883/./fig/lenses/Tile102030928RA0471339983832DECNEG0387957360796_filled.jpg" alt="Fig5.7" width="8%"/><img src="tmp_2611.15883/./fig/lenses/Tile102158828RA1956094527490DEC0655824562508_filled.jpg" alt="Fig5.8" width="8%"/><img src="tmp_2611.15883/./fig/lenses/RA145p8198694_DECp64p7264501_STCI_BGrestored_JYVIS_10x10arcsec_16bit.jpg" alt="Fig5.9" width="8%"/><img src="tmp_2611.15883/./fig/lenses/00945_Tile102028713RA0490026418080DECNEG0411026681570_STCI_Euclid_10x10arcsec.jpg" alt="Fig5.10" width="8%"/><img src="tmp_2611.15883/./fig/lenses/Tile102153310RA2571479841820DEC0576047031702_filled.jpg" alt="Fig5.11" width="8%"/><img src="tmp_2611.15883/./fig/lenses/Tile102009456RA0033210951000DECNEG0626182291000_asinh_vis_y.jpg" alt="Fig5.12" width="8%"/>

**Figure 5. -** Candidate lensed and lensing AGN, selected through crossmatches with external AGN catalogues and independent evidence of nuclear activity (Acevedo Barroso et al., in prep.; Moustakas et al., in prep.). (*fig:lqso*)

</div>
<div id="div_fig3">

<img src="tmp_2611.15883/./fig/lenses_uniform/spiral_final_100_correct.jpg" alt="Fig4.1" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102028157RA0418564021723DECNEG0416721071489_filled.jpg" alt="Fig4.2" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102035531RA0349730664844DECNEG0350039457858_filled.jpg" alt="Fig4.3" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/EUCL-DR1-L012444.jpg" alt="Fig4.4" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102020074RA0706063468315DECNEG0496163937709_filled.jpg" alt="Fig4.5" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/spiral2_mtf.jpg" alt="Fig4.6" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102031539RA0767484380079DECNEG0386112101678_filled.jpg" alt="Fig4.7" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102013101RA0146961863636DECNEG0572603836914_filled.jpg" alt="Fig4.8" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102164769RA2102653116781DEC0785038963034_filled.jpg" alt="Fig4.9" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102161578RA1842621676252DEC0707419320640_filled.jpg" alt="Fig4.10" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102090835RA0126861530727DEC0053775914704_mtf_vis_h.jpg" alt="Fig4.11" width="8%"/><img src="tmp_2611.15883/./fig/lenses_uniform/Tile102035576RA0622517897870DECNEG0347564288948_mtf_vis_h.jpg" alt="Fig4.12" width="8%"/>

**Figure 4. -** Striking examples of lensing by disc galaxies. (*fig:discs*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.15883"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\Sec}{Sect. }$
$\newcommand{\Fig}{Fig. }$
$\newcommand{\Eq}{Eq. }$
$\newcommand{ç}[1]{{\color{blue}{#1}}}$
$\newcommand{\rev}[1]{{\color{red}{#1}}}$
$\newcommand{\eprint}[2][]{$
$  \if!#1!#2\else#1\arxivprefixesep\ignorespaces#2\fi$
$}$
$\newcommand{\thefigure}{A.1}$
$\newcommand{\mydash}{-}$</div>



<div id="title">

# Euclid Data Release 1 (DR1): NISP spectroscopy of gravitational arcs and multiple images

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.15891-b31b1b.svg)](https://arxiv.org/abs/2611.15891)<mark>Appeared on: 2026-11-13</mark> -  _Paper intended for the A&A Special Issue "Euclid First Data Release". 15 pages, 5 figures_

</div>
<div id="authors">

E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark>

</div>
<div id="abstract">

**Abstract:** Applications of strong gravitational lensing in astrophysics andcosmology critically depend on building reliable strong lensing models, which require accurate measurements of on-sky positions of multiple images and of the redshifts of the background sources from which they originate. In this context, $\Euclid$ is a unique facility, as it simultaneously provides high-resolution multi-band photometric imaging and slitless spectroscopy for an unprecedentedly large sample of gravitational lenses, most of which are newly discovered. In this work, we exploit $\Euclid$ slitless spectroscopic data to determine the redshifts of four spatially extended lensed sources, and to reconstruct emission-line images.  Specifically, two of the target sources are lensed into giant arcs by the massive galaxy clusters SPT-CLJ0411 $-$ 4819 ( $z_\mathrm{d}=0.424$ , where "d" denotes the lensing deflector) and Abell 2280 ( $z_\mathrm{d}=0.326$ ). The third system is a quadruply imaged active galactic nucleus (AGN) candidate strongly lensed by a massive galaxy at $z_\mathrm{d}=0.68$ , while the fourth is a galaxy-scale Einstein ring produced by a lens galaxy at $z_\mathrm{d}=0.235$ . This work represents a proof of concept for the exploitation of $\Euclid$ slitless spectroscopic data in the study of strong gravitational lensing, providing one of the first determinations of spectroscopic redshifts for strongly lensed features from NISP spectra. These results demonstrate the feasibility of applying the same methodology to the very large sample of gravitational lenses that $\Euclid$ has started to discover and is expected to identify over the course of the mission.

</div>

<div id="div_fig1">

<img src="tmp_2611.15891/./figures/SPT-CLJ0411-4819.png" alt="Fig1" width="100%"/>

**Figure 1. -** \Euclid data of the galaxy cluster SPT-CLJ0411$-$4819 at $z_\mathrm{d}^\mathrm{sp}=0.424\pm0.006$, which lenses a background source with a measured \Euclid spectroscopic redshift of $z_\mathrm{s}^\mathrm{sp}=1.771\pm0.006$ into a giant gravitational arc. _ Bottom-left:_ RGB image of the cluster obtained by combining \Euclid photometric bands (\HE, \YE, \JE, and \IE) using the $\text$tt{azulero} code  (Basset_Azulero) . The spatial scale corresponding to \ang{;;5} refers to the deflector redshift. The green polygon corresponds to the aperture used to measure the magnitude of the gravitational arc in different bands and the associated uncertainty, as described in Appendix \ref{sc:PhotometricRedshift}. _ Bottom-right:_ Emission-line map at 1819.1 nm, showing the \ha emission associated with the arc. The map is obtained by combining 38 \Euclid dispersed spectroscopic images; the corresponding dispersion directions are indicated by the white segments in the bottom-right corner of the panel. The map, oversampled at the native pixel scale of the \IE filter (\ang{;;0.1}), is smoothed with a Gaussian kernel with a standard deviation of two pixels (\ang{;;0.2}). _ Top:_ Stacked \Euclid NISP-S spectra (in arbitrary units) extracted at two positions, "$\mathrm{Spec_1}$" and "$\mathrm{Spec_2}$’’, marked by cyan crosses in the RGB image. The spectral extraction is performed over a spatial aperture of three NISP pixels (corresponding to \ang{;;0.9}), oriented perpendicular to the dispersion direction of each exposure and centered on each extraction position.  (*fig:SPT-CLJ0411-4819*)

</div>
<div id="div_fig2">

<img src="tmp_2611.15891/./figures/J181607.9+681651.png" alt="Fig3" width="100%"/>

**Figure 3. -** Same as $\Fig$\ref{fig:SPT-CLJ0411-4819} for the WHJ181607.9$+$681651 system. The lens galaxy at $z_\mathrm{d}^\mathrm{ph}=0.68\pm0.04$, located at the centre of the RGB cutout in the bottom-left panel, produces four red multiple images of a background source with a measured \Euclid spectroscopic redshift of $z_\mathrm{s}^\mathrm{sp}=2.649\pm0.007$. Three emission lines, identified as the [$\ion${O}{iii}] doublet and possibly \hb, are detected in the \Euclid "$\mathrm{Spec_1}$" spectrum of the lensed source (black 1D-spectrum in the top panel). Only the stronger component of the [$\ion${O}{iii}] doublet is detected in "$\mathrm{Spec_2}$", displayed in light grey, at a lower signal-to-noise ratio. Both spectra were obtained by stacking three \Euclid spectroscopic exposures. The [$\ion${O}{iii}] emission from the four multiple images is visible in the emission-line map. (*fig:J181607*)

</div>
<div id="div_fig3">

<img src="tmp_2611.15891/./figures/Abell2280.png" alt="Fig2" width="100%"/>

**Figure 2. -** Same as $\Fig$\ref{fig:SPT-CLJ0411-4819} for the galaxy cluster Abell 2280 at $z_\mathrm{d}^\mathrm{sp}=0.326\pm0.002$. A source with a measured \Euclid spectroscopic redshift of $z_\mathrm{s}^\mathrm{sp}=1.670\pm0.012$ is lensed into the red giant arc visible in the RGB cutout in the bottom-left panel. For this system, the top panel shows a single spectrum, extracted at the position marked by the cyan cross in the bottom-left panel and obtained by stacking nine \Euclid spectroscopic exposures. (*fig:Abell2280*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.15891"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\url}[1]{\href{#1}{#1}}$
$\newcommand{\dodoi}[1]{doi:~\href{http://doi.org/#1}{\nolinkurl{#1}}}$
$\newcommand{\doeprint}[1]{\href{http://ascl.net/#1}{\nolinkurl{http://ascl.net/#1}}}$
$\newcommand{\doarXiv}[1]{\href{https://arxiv.org/abs/#1}{\nolinkurl{https://arxiv.org/abs/#1}}}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand{\myemail}{skywalker@galaxy.far.far.away}$
$\newcommand\natexlab{#1}$</div>



<div id="title">

# Desorption of water from cosmic silicates

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.16077-b31b1b.svg)](https://arxiv.org/abs/2611.16077)<mark>Appeared on: 2026-11-13</mark> - 

</div>
<div id="authors">

C. Jäger, D. Köhler, <mark>T. Henning</mark>, F. Langenhorst

</div>
<div id="abstract">

**Abstract:** Recent JWST and ALMA observations of protoplanetary disks have identified    water reservoirs in the inner regions of the disks. This would provide terrestrial planets forming therein access to a water reservoir. In situ formation of water molecules on grain surfaces was proposed as one possible formation process in the PDS 70 system, but transport by small silicate particles coupled to the gas could be another mechanism. Laboratory experiments on the thermally induced desorption of water molecules from different water-containing silicates at elevated temperatures provide another possibility for the enrichment of water molecules in the inner disk. Physically and chemically bound water in silicates can be released upon annealing processes. Water that is physically bound undergoes desorption at temperatures up to 473 K. In contrast, chemically bound water in form of different silanol or Mg-OH groups in amorphous silicates or phyllosilicates, can be released at higher temperatures, ranging from 773 to 923 K. The relevant temperatures are sufficient to trigger the solid-state recrystallization of amorphous silicates to Mg-rich orthopyroxene and olivine. The existence of these crystalline silicates has been substantiated through Spitzer and JWST-MIRI observations. Based on the experimental results, we postulate that the desorption of bound hydroxyl groups may serve as a mechanism for the formation of the water reservoir at temperatures up to 900 K, much  higher than the water sublimation temperature of 100 to 200 K.

</div>

<div id="div_fig1">

<img src="tmp_2611.16077/./f7.png" alt="Fig7" width="100%"/>

**Figure 7. -** A comparison of the desorption temperatures determined for the Serpentinite Erbendorf measured at  ambient pressure (blue curve) and 1$\times$10$^{-3}$ mbar (red curve, shifted by -10\%). (*Tg_vgl*)

</div>
<div id="div_fig2">

<img src="tmp_2611.16077/./f9.png" alt="Fig9" width="100%"/>

**Figure 9. -** DEGAS data of the Erbendorf serpentinite for m/z = 18 ($H_2$O) at three different heating rates (1 K, 5 K, and 10 K min$^{-1}$). The peak maxima shift clearly to higher temperature for higher heating rate.
 (*MS_2*)

</div>
<div id="div_fig3">

<img src="tmp_2611.16077/./f1.png" alt="Fig1" width="100%"/>

**Figure 1. -** Possible structural characteristics of chemisorbed water in amorphous magnesium silicates. It can be distinguished between isolated (1), geminal (2), vicinal (3), and internal hydroxyl groups in amorphous magnesium silicate. The dashed lines represent hydrogen bonds between adjacent surface OH groups. After the removal of chemisorbed OH, typical siloxan or bridging oxygen groups remain on the surface or internally. (*hydroxyl*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.16077"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\afme}[1]{ {\color{teal}(\it AE: #1)}}$
$\newcommand{\nnpz}{\texttt{NNPZ}}$
$\newcommand{\phosphoros}{\texttt{Phosphoros}}$
$\newcommand{\sfr}{\logten \mathrm{SFR}}$
$\newcommand{\logMsun}{\logten M_\odot}$
$\newcommand{\logMsunyr}{\logten(M_\odot \mathrm{yr}^{-1})}$
$\newcommand{\Mstarwun}{\logten(M_{\ast}/M_\odot)}$
$\newcommand{\sfrwun}{\logten (\mathrm{SFR}/M_\odot \mathrm{yr}^{-1})}$
$\newcommand{\orcid}[1]{ \orcidlink{#1}}$
$\newcommand{\linenumbers}[0]\usepackage$
$\newcommand{\arraystretch}{1.2}$</div>



<div id="title">

# Euclid Data Release 1 (DR1): The PHZ processing function for the physical properties of galaxies.

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.16444-b31b1b.svg)](https://arxiv.org/abs/2611.16444)<mark>Appeared on: 2026-11-13</mark> -  _13 pages, 8 figures. Paper intended for the A\&A Special Issue "Euclid First Data Release''_

</div>
<div id="authors">

E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark>

</div>
<div id="abstract">

**Abstract:** $\Euclid$ is a European Space Agency mission mapping the extragalactic sky, aiming to provide the most-accurate 3D view of the Universe at optical and near-infrared wavelengths, by measuring the photometric redshifts of billions of galaxies. Within the Euclid Consortium, this task is executed by the PHZ Organisation Unit. PHZ takes as input the multi-wavelength photometry of $\Euclid$ and ground-based data, and returns as output the source classification, photometric redshifts for the $\Euclid$ core science (i.e., cosmology), and photometric redshifts plus physical parameters and absolute magnitudes for non-cosmological science. As part of the first Euclid Data Release (DR1), we describe the details of the DR1 pipeline for non-cosmological science, along with the released data products, and their validation. The non-cosmological science data products are obtained with $\nnpz$ , a nearest-neighbour algorithm developed within PHZ that infers the properties of each target galaxy from its closest matches, in colour--magnitude space, within a reference sample of galaxies with known redshifts and physical parameters. We built this reference sample from $\Euclid$ observations of COSMOS, one of the Euclid Auxiliary Fields, labelling each galaxy with redshifts and physical parameters derived from the COSMOS2020 multi-wavelength photometry, consistently with the $\Euclid$ core-science results. Combined with the adjustments and priors required to generalise the results across the entire extragalactic sky, this approach yields high-quality and internally consistent data products. We tested the performance using both internal metrics and comparisons to ancillary data sets (e.g., COSMOS2020, 3D-HST, and DESI) and scaling relations (e.g., the star-forming main sequence); the results are in line with expectations and pre-DR1 simulations for both the Euclid Wide Survey and Euclid Deep Fields.

</div>

<div id="div_fig1">

<img src="tmp_2611.16444/./images/RefSample_WIDE_COSMOS_comparison.png" alt="Fig3.1" width="50%"/><img src="tmp_2611.16444/./images/RefSample_DEEP_COSMOS_comparison.png" alt="Fig3.2" width="50%"/>

**Figure 3. -** Comparison between the reference sample for the \gls{EWS}(first row) and \gls{EDF}(second row) used in \gls{dr1}. In the first column, the comparison is made with the subsample of reference sample sources that matched with the compilation of spectroscopic redshifts in COSMOS by 2026ApJS..282....6K. In the third column, the comparison is instead with the best-fit \texttt{LePhare} results for all the reference sample sources that matched with the COSMOS2020 Classic catalogue  (2022ApJS..258...11W) . The black line is the 1:1 relation, while the shaded grey area is the region beyond which a prediction is considered to be an outlier. The reported metrics are $\sigma_{\rm NMAD}$(purple), the outlier fraction, $f_{\rm out}$(blue), and the bias (green), all defined in Sect. \ref{sc:metrics}. The PIT (second columns) is measured as in Eq. (\ref{eq:pit}) for redshifts. The fourth column shows the difference \texttt{LePhare} minus \texttt{Phosphoros} photometric redshifts, normalized to the \texttt{LePhare} photo-$z$s, while $\Delta M_\ast$ is the difference between the \texttt{LePhare} and $\phosphoros$ best-fit measured logarithmic masses (see Sect. \ref{sc:Pipeline} for further details), and similarly for SFRs. The contours are the area containing $86\%$, $39\%$(corresponding to the $2\sigma$ and $1\sigma$ levels for a 2D histogram), and $20\%$ of the sample. (*fig:RefSample_C2020*)

</div>
<div id="div_fig2">

<img src="tmp_2611.16444/./images/EWS_WideLike.png" alt="Fig5" width="100%"/>

**Figure 5. -** Results for the EWS-like processing of EDF-N (first, second, and third column), which can be considered as a proxy for the \gls{EWS} in the northern hemisphere, and EDF-F (fourth and fifth column), which can be considered as a proxy for the \gls{EWS} in the southern hemisphere. Contours are similar to those in Fig. \ref{fig:RefSample_C2020}. (*fig:EWS_WideLike*)

</div>
<div id="div_fig3">

<img src="tmp_2611.16444/./images/EDFF_DEEP.png" alt="Fig6.1" width="50%"/><img src="tmp_2611.16444/./images/EDFN_DEEP.png" alt="Fig6.2" width="50%"/>

**Figure 6. -** Results for the Deep processing of EDF-F (*top panels*) and EDF-N (*bottom panels*). *Left*: comparison between the PHZ-PP measured photometric redshifts, and a compilation of spectroscopic redshifts (see Sect. \ref{sc:Data}). Other plots and lines as in Fig. \ref{fig:RefSample_C2020}. *Right*: like the centre panel of Fig. \ref{fig:EWS_WideLike}; for EDF-F the comparison is with the SED-fitting inferred \gls{sfr} in GOODS-S in 3D-HST; for EDF-N, with the \ha and \hb detected lines in DESI. (*fig:DeepFields*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.16444"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\linenumbers}[0]\begin$</div>



<div id="title">

# Euclid Data Release 1 (DR1): Multi-band photometry validation on COSMOS extragalactic sources

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.16825-b31b1b.svg)](https://arxiv.org/abs/2611.16825)<mark>Appeared on: 2026-11-13</mark> -  _Paper intended for the A\&A Special Issue "Euclid First Data Release''_

</div>
<div id="authors">

E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark>

</div>
<div id="abstract">

**Abstract:** The Euclid Data Release 1 (DR1) contains catalogues of sources covering roughly $1900 {\deg}^2$ , including deep coverage of legacy fields that enable an extensive comparison between $\Euclid$ imaging and multiwavelength observations from other instruments. The Cosmic Evolution Survey (COSMOS) field, observed by $\Euclid$ and included in DR1, has been observed with the $_ James Webb_$ Space Telescope, resulting in the COSMOS2025 catalogue.We compare the Euclid DR1 photometry with the multiwavelength COSMOS2025 data to assess the accuracy of the photometric measurements produced by the different methods implemented in the OU-MER pipeline. We use data from both the Euclid Wide and Deep Surveys, focussing exclusively on extragalactic sources.We use the fluxes of COSMOS2025 to predict the expected fluxes in the $\Euclid$ bands via spectral energy distribution fitting. The redshifts of the sources are either spectroscopic, for a sample of brighter sources, or photometric, to include fainter sources. Both the fluxes in the common Hyper Suprime-Cam bands and the SEDs from COSMOS2025 are used to validate the analysis and to characterise any biases or limitations of the method.The predicted fluxes and the OU-MER measurements are in good agreement, especially for bright and isolated sources. Median offsets depend on band, photometric method, survey depth, and redshift sample. Most are well within 10 \% , although the deep photo-z model-fitting measurements show NIR offsets of approximately 12–13 \% . However, complementary measurements on $\gaia$ stars show an opposite trend, highlighting that photometric validation approaches may suffer from intrinsic uncertainties that prevent a thorough assessment of systematic effects. We find that the OU-MER fluxes typically have larger offsets for systems that have not been correctly deblended in detection.Among the three methods, template-fitting photometry generally shows the smallest offset from the SED-based predictions in the NISP bands and toward fainter magnitudes, while aperture photometry and model-fitting are comparable in $\IE$ .Future analysis on deeper $\Euclid$ data and on fields at different Galactic latitudes will allow us to strengthen the constraints on the photometric accuracy of the $\Euclid$ surveys.

</div>

<div id="div_fig1">

<img src="tmp_2611.16825/./img_update/photoz_deep_wide_histograms_zphot_3.png" alt="Fig17.1" width="50%"/><img src="tmp_2611.16825/./img_update/photoz_deep_wide_histograms_zphot_3_COMMON.png" alt="Fig17.2" width="50%"/>

**Figure 17. -** Number of sources and median relative flux difference $\Delta F/F$ for the photo-$z$ sample, as a function of the \zphot forecast magnitudes in the four \Euclid filters. After the $\chi^2_{\rm red} < 2$ cut is applied, the sources are cross-matched either with the deep catalogue (top panel) or the wide one (second panel), represented in green.
    Then we only consider the sources with {\tt DEBLENDED\_FLAG}=0 (shown in red) and with ${\rm S/N} > 3$: this last cut selects a slightly different number of sources for each method and band (as seen in the histograms). The distributions are shown for SERSIC (brown), APHOT (pink), and TPHOT (gray). A reference horizontal line at $\Delta F/F=0$ is shown as a dashed gray line. The same plots are shown for the deep case (third panel) and for the wide case (bottom panel), but only for sources that have ${\rm S/N} > 3$ in all the bands and for all the three photometric methods (`common' histogram shown in brown). The same plots for the spec-$z$ sample are shown in Fig. \ref{fig:specz_deep_wide_histograms_zphot_3}. (*fig:photoz_deep_wide_histograms_zphot_3*)

</div>
<div id="div_fig2">

<img src="tmp_2611.16825/./img_update/specz_deep_histograms_zphot.png" alt="Fig8" width="100%"/>

**Figure 8. -** Distribution of sources in each \Euclid filter as a function of the \zphot forecast magnitude, for the spec-$z$ case. The histograms show the full initial sample of fitted sources (cyan), the sample after applying the $\chi_{\rm red}^2<4$ cut (orange), and the sample after cross-matching with the deep catalogue (green). Cross-matched sources are further divided into non-deblended (red) and deblended (purple) sources. Histograms for the very few NIR-detected sources are omitted, as they account for only a handful of objects. A similar plot for the photo-$z$ case is shown in Fig. \ref{fig:photoz_deep_histograms_zphot}. (*fig:specz_deep_histograms_zphot*)

</div>
<div id="div_fig3">

<img src="tmp_2611.16825/./img_update/offset_hsc_deep_0_1_specz_direct_cumulative.png" alt="Fig12.1" width="50%"/><img src="tmp_2611.16825/./img_update/offset_hsc_deep_0_1_specz_forecast_cumulative.png" alt="Fig12.2" width="50%"/>

**Figure 12. -** Same as Fig. \ref{fig:offset_euclid_0_1_specz} for the HSC bands, namely HSC $g$, $r$, $i$, and $z$. _Top_: Relative flux differences between the MER fluxes and the C25 ones in the four HSC bands for model-fitting (`SERSIC'), aperture photometry (`APHOT'), and template-fitting (`TPHOT'). Only the median lines with the NMADs for the sources with \texttt{VIS\_DET}=1, \texttt{DEBLENDED\_FLAG}=0, and with ${\rm S/N} > 3$ in MER are shown in these plots.
    _Bottom_: While the upper panel directly compares MER and C25, these plots show a comparison between the MER fluxes and the \zphot forecast. The largest difference is seen in the HSC $g$ band at the brighter end, while the other bands behave in the same way, within the errors, in the two panels. More details in Sect. \ref{sect:hsc_discussion}. The photo-$z$ case is shown in Fig. \ref{fig:offset_HSC_cosmos_mer_photoz}. (*fig:offset_HSC_cosmos_mer*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.16825"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\orcid}[1]{\orcidlink{#1}}$
$\newcommand{\JMASS}{\ensuremath{J_{\sfont{2MASS}}}\xspace}$
$\newcommand{\newcommandeq}{\triangleq}$
$\newcommand{\XSIR}{\ensuremath{X_{\text{SIR}}}}$
$\newcommand{\YSIR}{\ensuremath{Y_{\text{SIR}}}}$
$\newcommand{\d}{\ensuremath{\mathrm{d}}}$</div>



<div id="title">

# Euclid data release 1 (DR1): SIR spectroscopic calibration

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.17305-b31b1b.svg)](https://arxiv.org/abs/2611.17305)<mark>Appeared on: 2026-11-13</mark> -  _13 pages, 13 figures, submitted for publication to A&A_

</div>
<div id="authors">

E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark>

</div>
<div id="abstract">

**Abstract:** The $\Euclid$ space mission is dedicated to probing the nature of dark energy  and dark matter, through a comprehensive mapping of the Universe's  large-scale structure.  Central to these cosmological goals are the spectra  produced by the $\ac{NISP}$ and reduced by the scientific pipelines of the    SIR (Spectral near InfraRed) processing function.  Due to the intrinsic complexity of slitless spectroscopic data, scientific  observations cannot be self-calibrated by the science pipelines.  Consequently, the development of dedicated SIR calibration pipelines,  utilising specific pointing sequences and observation fields, is essential.  The resulting calibration products are fundamental to ensuring that reduced  spectra are characterised within requirements in both wavelength and flux.  This paper provides a detailed description of the SIR calibration pipelines  responsible for generating the products utilised by the science pipelines for $\ac{DR1}$ .  A significant challenge in these calibration tasks arises from the  requirement to model the entire focal plane as a unified entity.  Such a  global modelling approach is critical for maintaining astrometric and  spectroscopic continuity across the entire field of view. $\ac{DR1}$ provides more than 170 millions validated spectra in both red- and  blue-grism modes.  The accuracy of the reduced spectra is strictly dependent  on the performance of the calibration pipelines.  Comprehensive analyses of the calibration products and the resulting science  spectra demonstrate that current outputs successfully meet the mission  requirements.  Nevertheless, specific areas for optimisation have been  identified; ongoing development is focused on addressing these critical  points to further enhance data quality for future data releases.

</div>

<div id="div_fig1">

<img src="tmp_2611.17305/Figures/02_crv_opt.png" alt="Fig1" width="100%"/>

**Figure 1. -** Illustration of the OPT and CRV spectrometric models, jointly used
    to describe the spectrogram geometry.  *Upper panel*: a raw
    spectrogram, with an artificially enhanced curvature to facilitate the
    visualisation of the geometric distortions.  The red box is the area
    selected by the OPT calibration to compute the
    $(x_{\text{r}}, y_{\text{r}})$ spectrogram reference position (red dot);
    cross-dispersion offset $\delta y$ as a function of along-dispersion
    distance $\delta x$(yellow arrows) allow the characterisation of the
    spectrogram tilt and distortions.  *Lower left panel*: the
    $y_{\text{r}}$ coordinate is determined by fitting the cross-dispersion
    profile averaged over the red box along the dispersion axis.  A similar
    procedure is used to measure $\delta y$ along the trace
    (Sect. \ref{sec:CRV}).  *Lower right panel*: the $x_{\text{r}}$
    position is derived by fitting reference absorption spectral feature (here
    $\ion${Mg}{I}, see text) of the raw spectrum obtained by summing the red box
    along the cross-dispersion axis.  Both panels are from RGS000\texttt{+}0
    observation of a selected bright star ($\JMASS = 15.95$). (*fig:opt_crv_thumb*)

</div>
<div id="div_fig2">

<img src="tmp_2611.17305/Figures/01_ids_CALBLOCK_PV-009_R5_RGS000_fp.png" alt="Fig11" width="100%"/>

**Figure 11. -** Pattern of observation of the \ac{PN} in CALBLOCK-PV-009, with
      five distinct positions on each detector, following a quincunx pattern,
      for grism-tilt RGS000\texttt{+}0.  The target positions across the
      \ac{FP} during the 80 acquisitions are shown, along with mean (dot
      colour) and \acs*{RMS}(dot size) error of the global inverse dispersion
      solution (IDS), in pixels (see Sect. \ref{sec:IDS_DQC}).  The quincunx
      patterns on detectors 1x have been `compressed' so that corresponding
      zeroth-order spectrograms (to the left for RGS000\texttt{+}0) still fit
      within the \ac{FP}. (*fig:ids_pn_cross*)

</div>
<div id="div_fig3">

<img src="tmp_2611.17305/Figures/05_sensitivity_and_relative_error_allgrisms_plot_v4.png" alt="Fig4" width="100%"/>

**Figure 4. -** *Top panel*: sensitivity functions derived from \ac{PV}
    observations of the flux standard \ac{WD} GRW+70 5824, for the five
    grism-tilt configurations (see legend); vertical solid (resp. dotted) lines
    denote the 25\%(resp. 85\%) throughput masking boundaries, based on the
    local maxima at the two ends of the bandpass.  Spectral pixels outside the
    black lines are flagged as unusable (\protect\path{NOT_USE} bit) in the SIR
    pipeline, and those falling between the black and dotted lines are flagged
    as `suspicious' (\protect\path{ABS_FLUX} bit), since the throughput is
    falling rapidly and deemed uncertain there.  *Bottom panel*: relative
    error in the measured sensitivity functions for the five grism-tilt
    combinations. This plot represents a measure of the scatter in the observed
    optimally-extracted spectra of the bright \ac{WD} used for calibration. (*fig:sensfunctions*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.17305"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

150  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

21  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

10  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
